In [ ]:
drive.mount("/content/drive", force_remount=True)
import sys
sys.path.append("/content/drive/MyDrive/Crop Disease Detection Super Resolution")
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from training_utils import load_class_names, load_image_file_sample, graph_class_examples, data_summary, summarize_image_attributes
train_dir = "/content/drive/MyDrive/Crop Disease Detection Super Resolution/archive/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
class_names = load_class_names(train_dir)
print(class_names)
sample_image_files, sample_labels = load_image_file_sample(train_dir, class_names, 50)


In [ ]:

from torch.utils.data import DataLoader
import dataset_utils

dataset = dataset_utils.SuperResolutionDataset(sample_image_files)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

In [ ]:

#from training_loop import enhanced_training_loop
import importlib
from pathlib import Path
import os
import torch
import training_loop
import training_classes
import agricultural_sr
import generators
import discriminators
importlib.reload(training_loop)
importlib.reload(training_classes)
importlib.reload(agricultural_sr)
importlib.reload(discriminators)
importlib.reload(generators)

In [ ]:

def load_latest_generator_checkpoint(generator, 
                         checkpoint_dir="/content/drive/MyDrive/Crop Disease Detection Super Resolution/model_checkpoints", 
                         model_name="enhanced_model"):
    """Load the latest checkpoint if available"""
    
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir = checkpoint_dir / model_name
    latest_path = checkpoint_dir / "checkpoint_latest.pt"
    
    if not latest_path.exists():
        print("No checkpoint found at latest path.")
        return 0 # failed
    
    checkpoint = torch.load(latest_path)
    
    generator.load_state_dict(checkpoint['generator_state_dict'])
    
    print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch']}")
    
    return 1 # success 

In [ ]:

# input parameters
model_name = "test_model"
num_blocks = 8
total_epochs = 50
checkpoint_dir = "/content/drive/MyDrive/Crop Disease Detection Super Resolution/checkpoints"

model_name = f"{model_name}_{num_blocks}blocks_total{total_epochs}epochs"
if os.path.exists(os.path.join(checkpoint_dir, model_name)) == False:
    print(f"No checkpoint directory found for {model_name}, exiting...")

# replace with model of choice
generator = agricultural_sr.AgriculturalAttentionGenerator(num_residual_blocks=num_blocks).to(device)


# reload from latest checkpoint if available
status = load_latest_generator_checkpoint(generator, 
                                    checkpoint_dir=checkpoint_dir,
                                    model_name=model_name)

if status == 0:
    print("No checkpoint loaded, please use a trained model.")

In [ ]:
# Create an experiment where we evaluate the generator's impact on downstream classification
# Goal: assess if super-resolved images improve classification accuracy

# Want: accuracy/f1 score on low-res images vs super-resolved images vs some other baselines 
# Potential alternative baselines: bicubic upsampled images, denoised images, true high-res images

# Steps: 
# Generators create imagees from low-res inputs
# low-res image --> generator --> super-resolved image --> classification --> predicted class
# Train a classifier on image inputs and labels
# We will train multiple classifiers on different image inputs (low-res, super-resolved, bicubic, high-res)
# Compare models in terms of classification accuracy
# Use test images from dataset during evaluation
# State which model performs best and by how much
# Does super-resolution help classification?
# Does it perform better than bicubic upsampling?
# How close does it get to true high-res image classification performance?

# Hypothesis: Super-resolved images will yield higher classification accuracy than low-res and bicubic upsampled images
# approaching the performance of true high-res images.
